# Day 1: Hello, LLMs! — First Contact with Ollama
**Course:** LLM Engineering: Master AI and Large Language Models | Ed Donner (Udemy)
**Topic:** Introduction to Local LLMs with Ollama
**Models Used:** `llama3.2:1b`, `gemma3:4b` (no API key needed!)

Welcome to Day 1! Today we make our very first call to a Large Language Model (LLM) running entirely on our local machine — no API key, no cloud costs, just pure local AI.

## What You'll Learn
- What Ollama is and why it matters
- How to make your first LLM API call in Python
- Understanding the chat message structure (system/user/assistant roles)
- How to extract and parse the LLM's response
- Comparing two lightweight models: `llama3.2:1b` vs `gemma3:4b`

## What is Ollama?
**Ollama** is an open-source tool that allows you to run open-weight Large Language Models (LLMs) locally on your own machine. 

### Why run local LLMs?
1. **Cost:** Completely free. You don't pay per token.
2. **Privacy:** Your prompts and data never leave your computer.
3. **Offline:** No internet connection required once models are downloaded.
4. **Development Speed:** Rapid experimentation without API rate limits or network latency.

For this guide, we are using two lightweight models that run fast even on modest computers:
- **`llama3.2:1b`**: A 1.2-billion-parameter model from Meta. Great for fast chat and simple tasks.
- **`gemma3:4b`**: An extremely compact 270-million-parameter model from Google. Ideal for resource-constrained systems.

In [1]:
# Import the ollama library so we can talk to our local models
import ollama

print("Ollama library imported successfully!")

Ollama library imported successfully!


## Making an LLM Call
To talk to a local model, we use the function `ollama.chat()`.

We need to provide two inputs to this function:
1. `model`: The name of the model we want to use (in this case, `'llama3.2:1b'`).
2. `messages`: A list of messages we want to send. Each message is a simple dictionary with a `role` (who is speaking) and `content` (what they are saying).

In [2]:
messages = [{"role": "user", "content": "Introduce yourself in exactly two sentences."}]
# 1. Ask llama3.2:1b to introduce itself
response = ollama.chat(model="llama3.2:1b", messages=messages)

# 2. Extract the text response from the response object
reply = response.message.content

# 3. Print the answer to the screen
print(reply)
print(response)

I'm an artificial intelligence designed to assist and communicate with users in a helpful and informative way, and I'm constantly learning and improving to provide the best possible experience. I can provide information and entertainment on a wide range of topics, from science and history to entertainment and culture.
model='llama3.2:1b' created_at='2026-09-10T16:12:23.5780859Z' done=True done_reason='stop' total_duration=2644003200 load_duration=18083700 prompt_eval_count=33 prompt_eval_duration=277112000 eval_count=56 eval_duration=2325153000 message=Message(role='assistant', content="I'm an artificial intelligence designed to assist and communicate with users in a helpful and informative way, and I'm constantly learning and improving to provide the best possible experience. I can provide information and entertainment on a wide range of topics, from science and history to entertainment and culture.", thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None


In [ ]:
"""
ChatResponse
│
├── model
│     └── 'llama3.2:1b'
│
├── created_at
│     └── timestamp
│
├── done
│     └── True
│
├── done_reason
│     └── 'stop'
│
├── total_duration
│
├── load_duration
│
├── prompt_eval_count
│
├── prompt_eval_duration
│
├── eval_count
│
├── eval_duration
│
├── message ⭐
│     │
│     ├── role
│     │     └── 'assistant'
│     │
│     ├── content ⭐
│     │     └── "I'm an artificial intelligence..."
│     │
│     ├── thinking
│     │     └── None
│     │
│     ├── images
│     │     └── None
│     │
│     ├── tool_name
│     │     └── None
│     │
│     └── tool_calls
│           └── None
│
└── logprobs
      └── None
"""

## System Prompts: Giving the AI a Persona
By default, the AI acts as a helpful assistant. However, we can use the `'system'` role to change its behavior.

Let's test this by asking the same question: *"What is recursion?"* 
- In **Case A**, we won't use a system prompt.
- In **Case B**, we will tell the system: *"You are a grumpy pirate captain. Explain in under 30 words."*

In [5]:
# --- Case A: Default Behavior (No System Prompt) ---

print("Case A: Default Behavior")
print("-" * 50)

messages = [{"role": "user", "content": "What is a transformers in NLP?"}]

response_a = ollama.chat(model="llama3.2:3b", messages=messages)
print(response_a.message.content)

Case A: Default Behavior
--------------------------------------------------
In Natural Language Processing (NLP), a Transformer is a type of neural network architecture that is particularly well-suited for sequence-to-sequence tasks, such as machine translation, text summarization, and text classification. It was introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017.

The Transformer architecture is designed to handle sequential data, such as text, and is based on the concept of self-attention mechanisms, which allow the model to weigh the importance of different input elements when generating output.

The key components of the Transformer architecture are:

1. **Self-Attention Mechanism**: This is the core component of the Transformer architecture. It allows the model to attend to different parts of the input sequence simultaneously and weigh their importance. The self-attention mechanism is used to compute the attention weights, which are then used to compute 

In [10]:
messages = [
    {
        "role": "system",
        "content": "Explain NLP concepts simply, clearly, and with examples. Do not write code unless asked."
    },
    {
        "role": "user",
        "content": "What is a transformer in NLP?"
    }
]

response = ollama.chat(
    model="llama3.2:3b",
    messages=messages
)

print(response.message.content)

In Natural Language Processing (NLP), a transformer is a type of neural network architecture designed specifically for NLP tasks. It was introduced in 2017 and has since become a standard architecture for many NLP tasks.

The transformer architecture is based on two key concepts:

1. **Self-Attention Mechanism**: Instead of using traditional recurrent neural networks (RNNs) or convolutional neural networks (CNNs) to process sequences, the transformer uses self-attention to weigh the importance of different parts of the input sequence relative to each other. This allows the model to focus on the most relevant parts of the input sequence, rather than relying on sequential processing.
2. **Multi-Head Attention**: The transformer uses multiple attention mechanisms, each with a different set of weights, to capture different aspects of the input sequence. This is known as multi-head attention. Each attention head processes the input sequence independently, and the outputs are combined to for

## Multi-Turn Chat (Memory)
LLMs have no memory of past requests. If you tell the model: *"My name is John"* and then ask: *"What is my name?"* in a new request, it won't know!

To fix this, we create a list called `messages` and manually append each turn (both our questions and the AI's answers) to this list. Then we send the entire list to Ollama every time.

In [4]:
# 1. Initialize an empty message history list
messages = []

# TURN 1: Telling the AI our name
print("Turn 1: User says: 'Hi! My name is Arpan.'")

# Append user message
messages.append({'role': 'user', 'content': 'Hi! My name is Arpan.'})

# Call Ollama with the current messages list
response_1 = ollama.chat(model='llama3.2:1b', messages=messages)
reply_1 = response_1.message.content
print(f"Turn 1: AI responds: '{reply_1}'")

# Append assistant message to the history list
messages.append({'role': 'assistant', 'content': reply_1})


print("\n" + "="*60 + "\n")


# TURN 2: Asking the AI to remember our name
print("Turn 2: User says: 'What is my name?'")

# Append the new user question
messages.append({'role': 'user', 'content': 'What is my name?'})

# Call Ollama with the UPDATED messages list (containing history of Turn 1)
response_2 = ollama.chat(model='llama3.2:1b', messages=messages)
reply_2 = response_2.message.content
print(f"Turn 2: AI responds: '{reply_2}'")

# Append assistant message to the history list
messages.append({'role': 'assistant', 'content': reply_2})


print("\n" + "="*60 + "\n")


# Inspect the history list
print("Let's inspect what our 'messages' list looks like now:")
print("-" * 50)
for index, msg in enumerate(messages):
    print(f"[{index}] Role: {msg['role'].upper()} | Content: {msg['content']}")

Turn 1: User says: 'Hi! My name is Arpan.'
Turn 1: AI responds: 'Hello Arpan! How's it going? Is there something I can help you with or would you like to chat?'


Turn 2: User says: 'What is my name?'
Turn 2: AI responds: 'I can correct my mistake. You told me your name earlier, Arpan.'


Let's inspect what our 'messages' list looks like now:
--------------------------------------------------
[0] Role: USER | Content: Hi! My name is Arpan.
[1] Role: ASSISTANT | Content: Hello Arpan! How's it going? Is there something I can help you with or would you like to chat?
[2] Role: USER | Content: What is my name?
[3] Role: ASSISTANT | Content: I can correct my mistake. You told me your name earlier, Arpan.


## Model Comparison
Choosing the right model size is a core skill in AI engineering. 

We will compare:
1. **`llama3.2:1b`** (approx. 1.2 billion parameters)
2. **`gemma3:4b`** (approx. 4 billion parameters)

We will measure:
- **Latency:** How many seconds the model takes to respond.
- **Output:** How different the two models answer the same question.

In [7]:
import time

# Define the models we want to compare
models = ['llama3.2:1b', 'gemma3:4b']

# The question we will ask both models
prompt = "Explain what a AI is in one simple sentence."

for model_name in models:
    print(f"Testing model: {model_name}...")
    
    # Start the timer
    start_time = time.time()
    
    # Call the model
    response = ollama.chat(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}]
    )
    
    # Calculate time taken
    elapsed_time = time.time() - start_time
    
    # Get the text reply
    reply = response.message.content
    
    # Print results
    print(f"Time taken: {elapsed_time:.2f} seconds")
    print(f"Answer: {reply}")
    print("-" * 60)

Testing model: llama3.2:1b...
Time taken: 1.50 seconds
Answer: A computer program is an AI, or artificial intelligence, when it's designed to perform tasks that would typically require human intelligence, such as learning, problem-solving, and decision-making.
------------------------------------------------------------
Testing model: gemma3:4b...
Time taken: 3.12 seconds
Answer: Artificial intelligence is essentially a computer system designed to mimic human intelligence, performing tasks that typically require human thought processes like learning, problem-solving, and decision-making.
------------------------------------------------------------
